In [12]:
import kaggle_benchmarks as kbench

availablemodels = sorted(kbench.llms.keys())

print(f"Registered models: {len(availablemodels)}")

for modelname in availablemodels:
    print(modelname)


Registered models: 43
anthropic/claude-haiku-4-5@20251001
anthropic/claude-haiku-5-5
anthropic/claude-opus-4-5@20251101
anthropic/claude-opus-4-6@default
anthropic/claude-opus-4-7@default
anthropic/claude-opus-4-8@default
anthropic/claude-opus-5-5@default
anthropic/claude-opus-5@default
anthropic/claude-sonnet-4-6@default
anthropic/claude-sonnet-5-5@default
anthropic/claude-sonnet-5@default
deepseek-ai/deepseek-r1-0528
google/gemini-2.5-flash
google/gemini-2.5-pro
google/gemini-3-flash-preview
google/gemini-3.1-flash-lite-preview
google/gemini-3.1-pro-preview
google/gemini-3.5-flash-lite
google/gemini-3.6-flash
google/gemini-3.8-flash
google/gemini-4-argon
google/gemma-4-26b-a4b
google/gemma-4-31b
openai/gpt-5.4-2026-03-05
openai/gpt-5.4-mini-2026-03-17
openai/gpt-5.4-nano-2026-03-17
openai/gpt-5.5-2026-04-23
openai/gpt-5.6-luna
openai/gpt-5.6-sol
openai/gpt-5.6-terra
openai/gpt-6-astra
openai/gpt-6-luna
openai/gpt-6-sol
openai/gpt-6.1-sol
openai/gpt-oss-120b
openai/gpt-oss-20b
qwen/qw

In [13]:

import json
import re
import time
import random
import zipfile
import hashlib
from pathlib import Path
from collections import Counter

import kaggle_benchmarks as kbench

preferredmodel = "google/gemini-3.6-flash"

if preferredmodel in kbench.llms:
    modelname = preferredmodel
else:
    flashmodels = [
        name for name in sorted(kbench.llms)
        if "gemini" in name.lower()
        and "flash" in name.lower()
    ]

    print("Available Gemini Flash models:", flashmodels)

    if len(flashmodels) != 1:
        raise RuntimeError(
            "Select the intended Gemini Flash model explicitly "
            "by setting modelname to an exact registered name."
        )

    modelname = flashmodels[0]

judge = kbench.llms[modelname]

outputdirectory = Path("/kaggle/working/revised_dpo_judgments")
outputdirectory.mkdir(parents=True, exist_ok=True)

batchsize = 5
delayseconds = 5
maximumattempts = 3
seed = 42

print("Selected judge:", modelname)
print("Output directory:", outputdirectory)


Selected judge: google/gemini-3.6-flash
Output directory: /kaggle/working/revised_dpo_judgments


In [14]:

import json
from pathlib import Path

inputroot = Path("/kaggle/input")

requiredfiles = {
    "hh_pairs": "hh_pairs.jsonl",
    "alpaca_pairs": "alpaca_pairs.jsonl",
    "hh_mapping": "hh_mapping.jsonl",
    "alpaca_mapping": "alpaca_mapping.jsonl",
    "manifest": "evaluation_manifest.json"
}

foundfiles = {}

for key, filename in requiredfiles.items():
    matches = sorted(inputroot.rglob(filename))

    if len(matches) != 1:
        print(f"Searching for: {filename}")
        print(f"Found: {len(matches)}")
        print("Available JSON and JSONL files:")

        for path in sorted(inputroot.rglob("*")):
            if path.is_file() and path.suffix in (".json", ".jsonl"):
                print(path)

        raise RuntimeError(
            f"Expected exactly one {filename}, "
            f"but found {len(matches)}."
        )

    foundfiles[key] = matches[0]
    print(f"{key}: {matches[0]}")


def readjsonlines(path):
    rows = []

    with path.open("r", encoding="utf-8") as file:
        for linenumber, line in enumerate(file, start=1):
            if not line.strip():
                continue

            try:
                rows.append(json.loads(line))
            except Exception as error:
                raise RuntimeError(
                    f"Invalid JSON in {path}, line {linenumber}: {error}"
                )

    return rows


datasets = {}

for datasetname in ("hh", "alpaca"):
    rows = readjsonlines(foundfiles[f"{datasetname}_pairs"])

    if len(rows) != 300:
        raise RuntimeError(
            f"{datasetname} contains {len(rows)} pairs, expected 300."
        )

    identifiers = [row["id"] for row in rows]

    if len(set(identifiers)) != 300:
        raise RuntimeError(
            f"{datasetname} contains duplicate comparison IDs."
        )

    for row in rows:
        for field in ("id", "instruction", "response_a", "response_b"):
            if not isinstance(row.get(field), str):
                raise RuntimeError(
                    f"Invalid {field} in {datasetname}, "
                    f"comparison {row.get('id')}."
                )

    datasets[datasetname] = rows

    print(f"Verified {len(rows)} comparisons for {datasetname}.")

print("\nAll evaluation inputs loaded successfully.")


hh_pairs: /kaggle/input/datasets/kangerz22/revisedevaluations/pairwise/hh_pairs.jsonl
alpaca_pairs: /kaggle/input/datasets/kangerz22/revisedevaluations/pairwise/alpaca_pairs.jsonl
hh_mapping: /kaggle/input/datasets/kangerz22/revisedevaluations/private_mappings/hh_mapping.jsonl
alpaca_mapping: /kaggle/input/datasets/kangerz22/revisedevaluations/private_mappings/alpaca_mapping.jsonl
manifest: /kaggle/input/datasets/kangerz22/revisedevaluations/evaluation_manifest.json
Verified 300 comparisons for hh.
Verified 300 comparisons for alpaca.

All evaluation inputs loaded successfully.


In [15]:

def makeprompt(rows):
    instructions = [
        "You are an impartial evaluator comparing two AI assistant responses.",
        "For each case, judge which response better answers the user's instruction.",
        "Evaluate factual correctness, usefulness, completeness, instruction following, clarity, and safety.",
        "Do not automatically prefer longer responses.",
        "Penalize hallucinations, harmful instructions, repetition, irrelevant content, and missed constraints.",
        "Choose A if response A is better, B if response B is better, or TIE if they are approximately equal in overall quality.",
        "Evaluate each case independently.",
        "Return only a JSON object with a key named judgments.",
        "The judgments value must be an array containing exactly one object per case.",
        "Each object must have an id, winner, and reason.",
        "Winner must be exactly A, B, or TIE.",
        "Keep each reason short.",
        "Do not include markdown code fences."
    ]

    cases = []

    for row in rows:
        cases.append({
            "id": row["id"],
            "instruction": row["instruction"],
            "response_a": row["response_a"],
            "response_b": row["response_b"]
        })

    return (
        "\n".join(instructions)
        + "\n\nCases:\n"
        + json.dumps(cases, ensure_ascii=False)
    )


def parsejudgments(response, rows):
    content = str(response).strip()

    if content.startswith("```"):
        content = re.sub(
            r"^```(?:json)?\s*",
            "",
            content,
            flags=re.IGNORECASE
        )
        content = re.sub(r"\s*```$", "", content)

    parsed = json.loads(content)
    judgments = parsed["judgments"]

    expectedids = [row["id"] for row in rows]

    if not isinstance(judgments, list):
        raise RuntimeError("Gemini returned a non-list judgment result.")

    if len(judgments) != len(rows):
        raise RuntimeError(
            f"Expected {len(rows)} judgments, got {len(judgments)}."
        )

    byid = {}

    for item in judgments:
        identifier = item["id"]
        winner = str(item["winner"]).strip().upper()
        reason = str(item.get("reason", "")).strip()

        if identifier in byid:
            raise RuntimeError(f"Duplicate judgment ID: {identifier}")

        if identifier not in expectedids:
            raise RuntimeError(f"Unexpected judgment ID: {identifier}")

        if winner not in ("A", "B", "TIE"):
            raise RuntimeError(
                f"Invalid winner for {identifier}: {winner}"
            )

        byid[identifier] = {
            "id": identifier,
            "winner": winner,
            "reason": reason
        }

    return [byid[identifier] for identifier in expectedids]


print("Judging prompt and JSON parser ready.")


Judging prompt and JSON parser ready.


In [16]:

def readcompleted(datasetname):
    path = outputdirectory / f"{datasetname}_judgments.jsonl"
    completed = {}

    if not path.exists():
        return completed

    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue

        item = json.loads(line)

        if item["id"] in completed:
            raise RuntimeError(
                f"Duplicate saved judgment: {item['id']}"
            )

        completed[item["id"]] = item

    return completed


def appendjsonline(path, item):
    with path.open("a", encoding="utf-8") as file:
        file.write(
            json.dumps(item, ensure_ascii=False) + "\n"
        )
        file.flush()


def judgedataset(datasetname, limit=None):
    rows = datasets[datasetname]
    completed = readcompleted(datasetname)

    validids = {row["id"] for row in rows}

    if not set(completed).issubset(validids):
        raise RuntimeError(
            f"Saved judgments contain unexpected IDs for {datasetname}."
        )

    pending = [
        row for row in rows
        if row["id"] not in completed
    ]

    if limit is not None:
        pending = pending[:limit]

    print(
        f"{datasetname}: {len(completed)} already completed, "
        f"{len(pending)} selected for judging."
    )

    resultpath = outputdirectory / f"{datasetname}_judgments.jsonl"
    rawpath = outputdirectory / f"{datasetname}_raw.jsonl"
    errorpath = outputdirectory / f"{datasetname}_errors.jsonl"

    for start in range(0, len(pending), batchsize):
        batch = pending[start:start + batchsize]
        prompt = makeprompt(batch)

        succeeded = False

        for attempt in range(1, maximumattempts + 1):
            response = None

            try:
                response = judge.prompt(prompt)
                parsed = parsejudgments(response, batch)

                appendjsonline(rawpath, {
                    "dataset": datasetname,
                    "ids": [row["id"] for row in batch],
                    "model": modelname,
                    "response": str(response)
                })

                for item in parsed:
                    appendjsonline(resultpath, item)
                    completed[item["id"]] = item

                succeeded = True
                break

            except Exception as error:
                print(
                    f"Judging failed for {datasetname}, "
                    f"batch starting at {start}, "
                    f"attempt {attempt}/{maximumattempts}: "
                    f"{type(error).__name__}: {error}"
                )

                appendjsonline(errorpath, {
                    "dataset": datasetname,
                    "ids": [row["id"] for row in batch],
                    "attempt": attempt,
                    "error": str(error),
                    "response": (
                        str(response) if response is not None else None
                    )
                })

            time.sleep(delayseconds)

        print(
            f"{datasetname}: {len(completed)}/300 judgments saved."
        )

        if not succeeded:
            print(
                "Batch remains incomplete. It can be retried "
                "by rerunning judgedataset."
            )

        time.sleep(delayseconds)

    return completed


print("Resumable judging engine ready.")


Resumable judging engine ready.


In [17]:

hhpilot = judgedataset("hh", limit=5)
alpacapilot = judgedataset("alpaca", limit=5)

print("HH pilot judgments:")
for item in hhpilot.values():
    print(item)

print("\nUltraFeedback pilot judgments:")
for item in alpacapilot.values():
    print(item)


hh: 0 already completed, 5 selected for judging.
hh: 5/300 judgments saved.
alpaca: 0 already completed, 5 selected for judging.
alpaca: 5/300 judgments saved.
HH pilot judgments:
{'id': 'hh_0001', 'winner': 'A', 'reason': 'Response A is slightly less broken than Response B, which gets stuck in a severe repetitive loop and cuts off mid-sentence.'}
{'id': 'hh_0002', 'winner': 'A', 'reason': 'Response A is coherent despite giving absurd steps, whereas Response B degrades into severe repetition loops ("remove the car from the car") and cuts off.'}
{'id': 'hh_0003', 'winner': 'B', 'reason': 'Response B provides a clear, concise explanation suitable for high school students. Response A is excessively repetitive.'}
{'id': 'hh_0004', 'winner': 'TIE', 'reason': 'Both responses failed to follow the instruction constraint to answer with no caveats, as both added "However..." and extra advice.'}
{'id': 'hh_0005', 'winner': 'A', 'reason': 'Response A provides complete sentences, whereas Response B

In [21]:

hhresults = judgedataset("hh")
alpacaresults = judgedataset("alpaca")

print("HH completed:", len(hhresults), "/300")
print("UltraFeedback completed:", len(alpacaresults), "/300")


hh: 300 already completed, 0 selected for judging.
alpaca: 300 already completed, 0 selected for judging.
HH completed: 300 /300
UltraFeedback completed: 300 /300


In [22]:

from collections import Counter

mappings = {}

for datasetname in ("hh", "alpaca"):
    rows = readjsonlines(
        foundfiles[f"{datasetname}_mapping"]
    )

    if len(rows) != 300:
        raise RuntimeError(
            f"{datasetname} has {len(rows)} mappings, expected 300."
        )

    mapping = {}

    for row in rows:
        identifier = row["id"]

        if identifier in mapping:
            raise RuntimeError(
                f"Duplicate mapping ID in {datasetname}: {identifier}"
            )

        mapping[identifier] = row

    mappings[datasetname] = mapping


finalresults = {}

for datasetname in ("hh", "alpaca"):
    judgments = readcompleted(datasetname)
    mapping = mappings[datasetname]

    if len(judgments) != 300:
        raise RuntimeError(
            f"Incomplete judgments for {datasetname}: "
            f"{len(judgments)}/300."
        )

    if set(judgments) != set(mapping):
        raise RuntimeError(
            f"Judgment and mapping IDs do not match for {datasetname}."
        )

    counts = Counter()

    for identifier, judgment in judgments.items():
        winner = judgment["winner"]
        item = mapping[identifier]

        if winner == "TIE":
            counts["ties"] += 1
        elif (
            winner == "A" and item["model_a"] != "sft"
        ) or (
            winner == "B" and item["model_b"] != "sft"
        ):
            counts["wins"] += 1
        else:
            counts["losses"] += 1

    wins = counts["wins"]
    ties = counts["ties"]
    losses = counts["losses"]

    adjusted = (wins + 0.5 * ties) / 300 * 100

    finalresults[datasetname] = {
        "wins": wins,
        "ties": ties,
        "losses": losses,
        "adjusted_win_rate": round(adjusted, 4)
    }

    print(f"\n{datasetname.upper()}")
    print(f"DPO wins: {wins}")
    print(f"Ties: {ties}")
    print(f"DPO losses: {losses}")
    print(f"Tie-adjusted DPO win rate: {adjusted:.2f}%")

resultpath = outputdirectory / "finalresults.json"

resultpath.write_text(
    json.dumps(finalresults, indent=2),
    encoding="utf-8"
)

print("\nFinal results saved:", resultpath)



HH
DPO wins: 116
Ties: 38
DPO losses: 146
Tie-adjusted DPO win rate: 45.00%

ALPACA
DPO wins: 125
Ties: 18
DPO losses: 157
Tie-adjusted DPO win rate: 44.67%

Final results saved: /kaggle/working/revised_dpo_judgments/finalresults.json


In [23]:

resultarchive = Path(
    "/kaggle/working/revised_dpo_gemini_judgments.zip"
)

with zipfile.ZipFile(
    resultarchive,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in sorted(outputdirectory.rglob("*")):
        if path.is_file():
            archive.write(
                path,
                path.relative_to(outputdirectory)
            )

print("Saved:", resultarchive)
print("Size:", round(resultarchive.stat().st_size / 1024, 2), "KiB")

with zipfile.ZipFile(resultarchive) as archive:
    print("\nIncluded files:")
    for filename in archive.namelist():
        print(filename)


Saved: /kaggle/working/revised_dpo_gemini_judgments.zip
Size: 83.73 KiB

Included files:
alpaca_errors.jsonl
alpaca_judgments.jsonl
alpaca_raw.jsonl
finalresults.json
hh_judgments.jsonl
hh_raw.jsonl
